In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 필요하는 패키지 설치
!pip install -q adapter-transformers
!pip install -q --upgrade transformers accelerate
!pip install -q --upgrade transformers
!pip install -q peft==0.6.0
!pip install -q ipywidgets
!pip install -q datasets
!pip install -q --upgrade BitsAndBytes
!pip install -q fire

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.8 MB/s eta 0:00:0

# Library

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
from typing import List
import json

# 운영체제
import os
import sys

# 디버깅 전용
import inspect

# 딥러닝
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

# Llama 모델
import transformers
from datasets import load_dataset, concatenate_datasets
from transformers import EarlyStoppingCallback
from transformers import LlamaForCausalLM, LlamaTokenizerFast, LlamaTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, DataCollatorForSeq2Seq

# QLoRA
from peft import (  # noqa: E402
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_int8_training,
    set_peft_model_state_dict,
)
from transformers import BitsAndBytesConfig

# 평가지표
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

In [ ]:
%cd /content/drive/MyDrive/Project/

/content/drive/MyDrive/Project


# Utils

In [ ]:
# Prompt 생성함수
def generate_prompt(data_point):
    # sorry about the formatting disaster gotta move fast
    if data_point["input"]:
        return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{data_point["instruction"]}

### Input:
{data_point["input"]}

### Response:
{data_point["output"]}"""
    else:
        return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{data_point["instruction"]}

### Response:
{data_point["output"]}"""

In [ ]:
# LLaMA 모델 Training 함수

def train(
    # model/data params
    base_model: str = "",  # the only required argument
    train_data_path: str = "/content/drive/MyDrive/Amazon_Dataset/Preprocessed/grocery_and_gourmet_food_train_prompt_2022.json",
    val_data_path: str = "/content/drive/MyDrive/Amazon_Dataset/Preprocessed/grocery_and_gourmet_food_valid_prompt_2022.json",
    test_data_path: str = "/content/drive/MyDrive/Amazon_Dataset/Preprocessed/grocery_and_gourmet_food_test_prompt_2022.json",
    output_dir: str = "./lora-alpaca",
    train_sample: int = 256,
    valid_sample: int = 10000,
    test_sample: int = 10000,
    seed: int = 0,
    # training hyperparams
    batch_size: int = 128,
    micro_batch_size: int = 4,
    num_epochs: int = 3,
    learning_rate: float = 3e-4,
    cutoff_len: int = 256,
    # lora hyperparams
    lora_r: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.05,
    lora_target_modules: List[str] = [
        "q_proj",
        "v_proj",
    ],
    # llm hyperparams
    train_on_inputs: bool = True,  # if False, masks out inputs in loss
    group_by_length: bool = False,  # faster, but produces an odd training loss curve
    # wandb params
    wandb_project: str = "",
    wandb_run_name: str = "",
    wandb_watch: str = "",  # options: false | gradients | all
    wandb_log_model: str = "",  # options: false | true
    resume_from_checkpoint: str = None,  # either training checkpoint or final adapter

):
    print(
        f"Training Alpaca-LoRA model with params:\n"
        f"base_model: {base_model}\n"
        f"train_data_path: {train_data_path}\n"
        f"val_data_path: {val_data_path}\n"
        f"train_sample: {train_sample}\n"
        f"valid_sample: {valid_sample}\n"
        f"test_sample: {test_sample}\n"
        f"seed: {seed}\n"
        f"output_dir: {output_dir}\n"
        f"batch_size: {batch_size}\n"
        f"micro_batch_size: {micro_batch_size}\n"
        f"num_epochs: {num_epochs}\n"
        f"learning_rate: {learning_rate}\n"
        f"cutoff_len: {cutoff_len}\n"
        f"lora_r: {lora_r}\n"
        f"lora_alpha: {lora_alpha}\n"
        f"lora_dropout: {lora_dropout}\n"
        f"lora_target_modules: {lora_target_modules}\n"
        f"train_on_inputs: {train_on_inputs}\n"
        f"group_by_length: {group_by_length}\n"
        f"wandb_project: {wandb_project}\n"
        f"wandb_run_name: {wandb_run_name}\n"
        f"wandb_watch: {wandb_watch}\n"
        f"wandb_log_model: {wandb_log_model}\n"
        f"resume_from_checkpoint: {resume_from_checkpoint}\n"
    )
    assert (
        base_model
    ), "Please specify a --base_model, e.g. --base_model='decapoda-research/llama-7b-hf'"
    gradient_accumulation_steps = batch_size // micro_batch_size
    # print(f"gradient_accumulation_steps: {gradient_accumulation_steps}")

    device_map = "auto"
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    ddp = world_size != 1
    if ddp:
        device_map = {"": int(os.environ.get("LOCAL_RANK") or 0)}
        gradient_accumulation_steps = gradient_accumulation_steps // world_size

    # Check if parameter passed or if set within environ
    use_wandb = len(wandb_project) > 0 or (
        "WANDB_PROJECT" in os.environ and len(os.environ["WANDB_PROJECT"]) > 0
    )
    # Only overwrite environ if wandb param passed
    if len(wandb_project) > 0:
        os.environ["WANDB_PROJECT"] = wandb_project
    if len(wandb_watch) > 0:
        os.environ["WANDB_WATCH"] = wandb_watch
    if len(wandb_log_model) > 0:
        os.environ["WANDB_LOG_MODEL"] = wandb_log_model


    # 1. 양자화 설정 (BitsAndBytesConfig)
    # QLoRA 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_enable_fp32_cpu_offload=False,
    )

    model = LlamaForCausalLM.from_pretrained(
        base_model,
        torch_dtype=torch.float16,
        device_map=device_map,
        quantization_config=bnb_config,
        cache_dir='./llama3.1/',
        token = '[REDACTED]'
    )

    tokenizer = LlamaTokenizerFast.from_pretrained(base_model, cache_dir='./llama3.2/', token = '[REDACTED]')

    tokenizer.pad_token_id = (
        0  # unk. we want this to be different from the eos token
    )
    tokenizer.padding_side = "left"  # Allow batched inference

    def tokenize(prompt, add_eos_token=True):
        # there's probably a way to do this with the tokenizer settings
        # but again, gotta move fast
        result = tokenizer(
            prompt,
            truncation=True,
            max_length=cutoff_len,
            padding=True,
            return_tensors=None,
        )
        if (
            result["input_ids"][-1] != tokenizer.eos_token_id
            and len(result["input_ids"]) < cutoff_len
            and add_eos_token
        ):
            result["input_ids"].append(tokenizer.eos_token_id)
            result["attention_mask"].append(1)

        result["labels"] = result["input_ids"].copy()

        return result

    def generate_and_tokenize_prompt(data_point):
        full_prompt = generate_prompt(data_point)
        tokenized_full_prompt = tokenize(full_prompt)

        if not train_on_inputs:
            user_prompt = generate_prompt({**data_point, "output": ""})
            tokenized_user_prompt = tokenize(user_prompt, add_eos_token=False)
            user_prompt_len = len(tokenized_user_prompt["input_ids"])

            tokenized_full_prompt["labels"] = [
                -100
            ] * user_prompt_len + tokenized_full_prompt["labels"][
                user_prompt_len:
            ]  # could be sped up, probably
        return tokenized_full_prompt

    model = prepare_model_for_int8_training(model)

    config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=lora_target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, config)

    if train_data_path.endswith(".json"):  # todo: support jsonl
        train_data = load_dataset("json", data_files=train_data_path)
    else:
        train_data = load_dataset(train_data_path)

    if val_data_path.endswith(".json"):  # todo: support jsonl
        val_data = load_dataset("json", data_files=val_data_path)
    else:
        val_data = load_dataset(val_data_path)

    if test_data_path.endswith(".json"):  # todo: support jsonl
        test_data = load_dataset("json", data_files=test_data_path)
    else:
        test_data = load_dataset(test_data_path)

    # train_data = train_data.shuffle(seed=42)[:sample] if sample > -1 else train_data
    # print(len(train_data))
    if resume_from_checkpoint:
        # Check the available weights and load them
        checkpoint_name = os.path.join(
            resume_from_checkpoint, "pytorch_model.bin"
        )  # Full checkpoint
        if not os.path.exists(checkpoint_name):
            checkpoint_name = os.path.join(
                resume_from_checkpoint, "adapter_model.bin"
            )  # only LoRA model - LoRA config above has to fit
            resume_from_checkpoint = (
                False  # So the trainer won't try loading its state
            )
        # The two files above have a different name depending on how they were saved, but are actually the same.
        if os.path.exists(checkpoint_name):
            print(f"Restarting from {checkpoint_name}")
            adapters_weights = torch.load(checkpoint_name)
            model = set_peft_model_state_dict(model, adapters_weights)
        else:
            print(f"Checkpoint {checkpoint_name} not found")

    model.print_trainable_parameters()  # Be more transparent about the % of trainable params.

    train_data["train"] = train_data["train"].shuffle(seed=seed).select(range(train_sample)) if train_sample > -1 else train_data["train"].shuffle(seed=seed)
    val_data["train"] = val_data["train"].shuffle(seed=seed).select(range(valid_sample)) if valid_sample > -1 else val_data["train"].shuffle(seed=seed)
    test_data["train"] = test_data["train"].shuffle(seed=seed).select(range(valid_sample)) if valid_sample > -1 else val_data["train"].shuffle(seed=seed)

    # data_tokenize
    train_data = (train_data["train"].map(generate_and_tokenize_prompt))
    val_data = (val_data["train"].map(generate_and_tokenize_prompt))
    test_data = (test_data["train"].map(generate_and_tokenize_prompt))

    if not ddp and torch.cuda.device_count() > 1:
        model.is_parallelizable = True
        model.model_parallel = True

    def preprocess_logits_for_metrics(logits, labels):
        """
        Original Trainer may have a memory leak.
        This is a workaround to avoid storing too many tensors that are not needed.
        """

        YES_ID = tokenizer.convert_tokens_to_ids("Yes")
        NO_ID = tokenizer.convert_tokens_to_ids("No")

        labels_index = torch.argwhere(torch.bitwise_or(labels == YES_ID, labels == NO_ID))
        if labels_index.shape[0] == 0:
            return torch.tensor([]), torch.tensor([])

        gold = torch.where(labels[labels_index[:, 0], labels_index[:, 1]] == NO_ID, 0, 1)

        labels_index[:, 1] = labels_index[:, 1] - 1  # shift for decoder step

        logits = logits.softmax(dim=-1)
        logits = torch.softmax(logits[labels_index[:, 0], labels_index[:, 1]][:, [NO_ID, YES_ID]], dim=-1)

        return logits[:, 1][2::3], gold[2::3]

    def compute_metrics(eval_preds):

        # label 기반으로 auc 측정
        pre, labels = eval_preds
        auc = roc_auc_score(pre[1], pre[0])

        return {"auc": auc}

    os.environ["WANDB_DISABLED"] = "true"

    if train_sample > -1:
        if train_sample <= 128 :
            eval_step = 10
        else:
            eval_step = train_sample / 128 * 5

    trainer = transformers.Trainer(
        model=model,
        train_dataset=train_data,
        eval_dataset=val_data,
        args=transformers.TrainingArguments(
            per_device_train_batch_size=micro_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            warmup_steps=20,
            num_train_epochs=num_epochs,
            learning_rate=learning_rate,
            fp16=True,
            logging_steps=8,
            optim="adamw_torch",
            logging_strategy="epoch",
            eval_strategy="epoch",
            save_strategy="epoch",
            eval_steps=eval_step,
            save_steps=eval_step,
            output_dir=output_dir,
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_auc",
            ddp_find_unused_parameters=False if ddp else None,
            group_by_length=group_by_length,
            report_to=None,
            # report_to="wandb" if use_wandb else None,
            # run_name=wandb_run_name if use_wandb else None,
            # eval_accumulation_steps=10,
        ),
        data_collator=transformers.DataCollatorForSeq2Seq(
            tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
        ),
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        callbacks = [EarlyStoppingCallback(early_stopping_patience=10)]
    )
    model.config.use_cache = False

    old_state_dict = model.state_dict
    model.state_dict = (
        lambda self, *_, **__: get_peft_model_state_dict(
            self, old_state_dict()
        )
    ).__get__(model, type(model))

    if torch.__version__ >= "2" and sys.platform != "win32":
        model = torch.compile(model)

    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    model.save_pretrained(output_dir)

    return (trainer.predict(test_data), test_data['uid'])


In [ ]:
# 데이터셋의 uid를 기반으로 정렬해서 평가함. (일반 추천 성능을 재확인)
# Area Under Curve의 성능 평가 방식과 동일함. (순차 추천 성능 아님)
# Next Sequence Prediction을 기반으로 평가하지 않아서, 순차 예측에 대한 성능을 구할 수 없음.
# Test 데이터 프레임에서 예측 확률 순서대로 Precision@K, Recall@K, F1-Score@K, NDCG@K 확인

def compute_ranking_metrics(pred_probs, gold, user_ids, k=10):
    """
    각 사용자별로 예측 점수와 정답을 그룹화한 후,
    Precision@K, Recall@K, F1@K, NDCG@K, Hit@K, 그리고 사용자 평균 AUC(uAUC)를 계산합니다.

    :param pred_probs: 전체 샘플의 예측 확률 (1D numpy array)
    :param gold: 전체 샘플의 정답 레이블 (binary, 1D numpy array)
    :param user_ids: 전체 샘플의 사용자 id (1D numpy array)
    :param k: K 값
    :return: dict, 각 지표의 평균 값.
    """
    unique_users = np.unique(user_ids)
    precision_list, recall_list, f1_list = [], [], []
    ndcg_list, hit_list, user_auc_list = [], [], []

    for uid in unique_users:
        # 해당 사용자에 해당하는 인덱스 추출
        idx = np.where(user_ids == uid)[0]
        user_preds = pred_probs[idx]
        user_gold = gold[idx]

        # 예측 확률에 따라 내림차순 정렬
        sorted_indices = np.argsort(-user_preds)
        sorted_gold = user_gold[sorted_indices]

        # k_actual: 해당 사용자에 대해 사용할 실제 k 값 (사용자 샘플 수보다 크면, 샘플 수로 제한)
        k_actual = min(k, len(sorted_gold))

        # Precision@k: 상위 k개에서 양성(1)인 비율 (여기서는 k 고정)
        precision = np.sum(sorted_gold[:k] > 0) / float(k)
        precision_list.append(precision)

        # Recall@k: 상위 k개에서 양성의 개수를 전체 양성 수로 나눈 값
        total_positives = np.sum(user_gold > 0)
        recall = np.sum(sorted_gold[:k] > 0) / float(total_positives) if total_positives > 0 else 0
        recall_list.append(recall)

        # F1@k: Precision과 Recall의 조화 평균
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        f1_list.append(f1)

        # NDCG@k: DCG와 이상적인 DCG의 비율로 계산 (binary relevance 사용)
        # k_actual을 사용하여, 실제 샘플 수에 맞게 계산합니다.
        dcg = np.sum((2**sorted_gold[:k_actual] - 1) / np.log2(np.arange(2, k_actual + 2)))
        ideal_gold = np.sort(user_gold)[::-1]
        idcg = np.sum((2**ideal_gold[:k_actual] - 1) / np.log2(np.arange(2, k_actual + 2))) if np.sum(user_gold) > 0 else 0
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg)

        # Hit Rate@k: 상위 k개에 양성이 하나라도 있으면 hit = 1, 아니면 0
        hit = 1.0 if np.sum(sorted_gold[:k] > 0) > 0 else 0.0
        hit_list.append(hit)

        # 사용자별 AUC (uAUC)
        if len(np.unique(user_gold)) == 2:
            user_auc = roc_auc_score(user_gold, user_preds)
            user_auc_list.append(user_auc)

    uauc = np.mean(user_auc_list) if len(user_auc_list) > 0 else 0

    return {
        f"precision@{k}": np.mean(precision_list),
        f"recall@{k}": np.mean(recall_list),
        f"f1@{k}": np.mean(f1_list),
        f"ndcg@{k}": np.mean(ndcg_list),
        f"hit@{k}": np.mean(hit_list),
        "uauc": uauc
    }

# Run LLM(meta-llama/Llama-3.1-8B-Instruct)
- Epoch은 3으로 수행했습니다.

## Electronics 성능

In [ ]:
# test 데이터의 확률값 반환
test_predict = train("meta-llama/Llama-3.1-8B-Instruct")

Training Alpaca-LoRA model with params:
base_model: meta-llama/Llama-3.1-8B-Instruct
train_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/electronics_train_prompt_2022.json
val_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/electronics_valid_prompt_2022.json
train_sample: 256
valid_sample: 10000
test_sample: 10000
seed: 0
output_dir: ./lora-alpaca
batch_size: 128
micro_batch_size: 4
num_epochs: 3
learning_rate: 0.0003
cutoff_len: 256
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules: ['q_proj', 'v_proj']
train_on_inputs: True
group_by_length: False
wandb_project: 
wandb_run_name: 
wandb_watch: 
wandb_log_model: 
resume_from_checkpoint: None



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'LlamaTokenizerFast'.
/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:135: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.04241987003816259


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Auc
1,3.011000,3.007095,0.971461
2,3.003000,2.984019,0.971572
3,2.970300,2.940887,0.971736


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [ ]:
# 사용자별 리스트를 만들고 높은 확률 순으로 정렬한 후 성능 측정
K = [1 if i == 0 else 5 * i for i in range(0, 20)]

for k in K:
    print(compute_ranking_metrics(test_predict[0][0][0], test_predict[0][0][1], test_predict[1], k=k))

{'precision@1': np.float64(0.5784960422163589), 'recall@1': np.float64(0.5634839489885665), 'f1@1': np.float64(0.5682937554969217), 'ndcg@1': np.float64(0.5784960422163589), 'hit@1': np.float64(0.5784960422163589), 'uauc': np.float64(0.9755639097744361)}
{'precision@5': np.float64(0.12240545294635005), 'recall@5': np.float64(0.5794854881266491), 'f1@5': np.float64(0.20066905390124393), 'ndcg@5': np.float64(0.5790938357684243), 'hit@5': np.float64(0.5794854881266491), 'uauc': np.float64(0.9755639097744361)}
{'precision@10': np.float64(0.061202726473175026), 'recall@10': np.float64(0.5794854881266491), 'f1@10': np.float64(0.11019532358582491), 'ndcg@10': np.float64(0.5790938357684243), 'hit@10': np.float64(0.5794854881266491), 'uauc': np.float64(0.9755639097744361)}
{'precision@15': np.float64(0.040801817648783344), 'recall@15': np.float64(0.5794854881266491), 'f1@15': np.float64(0.07597375500679372), 'ndcg@15': np.float64(0.5790938357684243), 'hit@15': np.float64(0.5794854881266491), 'u

## Toys and Games 성능

In [ ]:
# test 데이터의 확률값 반환
test_predict = train("meta-llama/Llama-3.1-8B-Instruct")

Training Alpaca-LoRA model with params:
base_model: meta-llama/Llama-3.1-8B-Instruct
train_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/toys_and_games_train_prompt_2022.json
val_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/toys_and_games_valid_prompt_2022.json
train_sample: 256
valid_sample: 10000
test_sample: 10000
seed: 0
output_dir: ./lora-alpaca
batch_size: 128
micro_batch_size: 4
num_epochs: 3
learning_rate: 0.0003
cutoff_len: 256
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules: ['q_proj', 'v_proj']
train_on_inputs: True
group_by_length: False
wandb_project: 
wandb_run_name: 
wandb_watch: 
wandb_log_model: 
resume_from_checkpoint: None



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'LlamaTokenizerFast'.


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

... (3 outputs left out)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Auc
1,3.245300,3.277192,0.973694
2,3.235200,3.255014,0.973520
3,3.202500,3.213399,0.973610


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [ ]:
# 사용자별 리스트를 만들고 높은 확률 순으로 정렬한 후 성능 측정
K = [1 if i == 0 else 5 * i for i in range(0, 20)]

for k in K:
    print(compute_ranking_metrics(test_predict[0][0][0], test_predict[0][0][1], test_predict[1], k=k))

{'precision@1': np.float64(0.6160458452722063), 'recall@1': np.float64(0.5713190025510942), 'f1@1': np.float64(0.5849542291519368), 'ndcg@1': np.float64(0.6160458452722063), 'hit@1': np.float64(0.6160458452722063), 'uauc': np.float64(0.9711398928011307)}
{'precision@5': np.float64(0.14516801250325603), 'recall@5': np.float64(0.6187596389602121), 'f1@5': np.float64(0.22993610612131066), 'ndcg@5': np.float64(0.6177700412938919), 'hit@5': np.float64(0.6190414170356864), 'uauc': np.float64(0.9711398928011307)}
{'precision@10': np.float64(0.07276634540244856), 'recall@10': np.float64(0.6190414170356864), 'f1@10': np.float64(0.12820123595220229), 'ndcg@10': np.float64(0.617816235095293), 'hit@10': np.float64(0.6190414170356864), 'uauc': np.float64(0.9711398928011307)}
{'precision@15': np.float64(0.0485108969349657), 'recall@15': np.float64(0.6190414170356864), 'f1@15': np.float64(0.08892317653351955), 'ndcg@15': np.float64(0.617816235095293), 'hit@15': np.float64(0.6190414170356864), 'uauc':

# Grocery and Gourmet Food 성능

In [ ]:
# test 데이터의 확률값 반환
test_predict = train("meta-llama/Llama-3.1-8B-Instruct")

Training Alpaca-LoRA model with params:
base_model: meta-llama/Llama-3.1-8B-Instruct
train_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/grocery_and_gourmet_food_train_prompt_2022.json
val_data_path: /content/drive/MyDrive/Amazon_Dataset/Preprocessed/grocery_and_gourmet_food_valid_prompt_2022.json
train_sample: 256
valid_sample: 10000
test_sample: 10000
seed: 0
output_dir: ./lora-alpaca
batch_size: 128
micro_batch_size: 4
num_epochs: 3
learning_rate: 0.0003
cutoff_len: 256
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target_modules: ['q_proj', 'v_proj']
train_on_inputs: True
group_by_length: False
wandb_project: 
wandb_run_name: 
wandb_watch: 
wandb_log_model: 
resume_from_checkpoint: None



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'LlamaTokenizerFast'.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:135: FutureWarning: prepare_model_for_int8_training is deprecated and will be 

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

... (3 outputs left out)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Auc
1,2.986000,2.972801,0.967528
2,2.978300,2.948578,0.967446
3,2.943900,2.902974,0.967356


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [ ]:
# 사용자별 리스트를 만들고 높은 확률 순으로 정렬한 후 성능 측정
K = [1 if i == 0 else 5 * i for i in range(0, 20)]

for k in K:
    print(compute_ranking_metrics(test_predict[0][0][0], test_predict[0][0][1], test_predict[1], k=k))

{'precision@1': np.float64(0.6217973728703342), 'recall@1': np.float64(0.5790312096106088), 'f1@1': np.float64(0.5917989934084356), 'ndcg@1': np.float64(0.6217973728703342), 'hit@1': np.float64(0.6217973728703342), 'uauc': np.float64(0.9711560636361998)}
{'precision@5': np.float64(0.14657302640135259), 'recall@5': np.float64(0.6254590783560741), 'f1@5': np.float64(0.23171606575820394), 'ndcg@5': np.float64(0.6242604429637423), 'hit@5': np.float64(0.6258291065158018), 'uauc': np.float64(0.9711560636361998)}
{'precision@10': np.float64(0.07354662504877098), 'recall@10': np.float64(0.62575574163557), 'f1@10': np.float64(0.1292778147992476), 'ndcg@10': np.float64(0.6242936014845382), 'hit@10': np.float64(0.6258291065158018), 'uauc': np.float64(0.9711560636361998)}
{'precision@15': np.float64(0.04910044652533922), 'recall@15': np.float64(0.6258291065158018), 'f1@15': np.float64(0.08977574676914302), 'ndcg@15': np.float64(0.6242936014845382), 'hit@15': np.float64(0.6258291065158018), 'uauc':